In [0]:
%sql
create connection if not exists youtube_earthquake_conn
type HTTP
options
(
host = "https://earthquake.usgs.gov",
base_path = "/earthquakes/feed/v1.0",
port= 443,
bearer_token = 'na'
)

https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson

In [0]:
# Importing data rest api( But this a harcoded value we are using)
import requests
import json
url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson'
response = requests.get(url)
response.json()

In [0]:
# Need to import the connection details from Databricks under Unity catalog
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn = w.connections.get("youtube_earthquake_conn")
#print(conn) 
base_url = f"{conn.options['host']}{conn.options['base_path']}"
print(base_url)


In [0]:
%sql
-- Creating volume inside bronze schema so as to ingest data from connection(These are again hardcoded values)
use catalog youtube_dev;
use schema bronze;
create volume if not exists earthquake_data;

In [0]:
dbutils.widgets.text('catalog_name','youtube_dev')
catalog_name = dbutils.widgets.get('catalog_name')
#print(catalog_name)

In [0]:
# WE were hardcoing in above code we can use widget to get the value
spark.sql(f"use catalog {catalog_name}")
spark.sql("use schema bronze")
spark.sql("create volume if not exists earthquake_data")

In [0]:
import requests
import json
url = f'{base_url}/summary/all_day.geojson'
response = requests.get(url)
response.json()

In [0]:
# Dump data into volume
# Used Catalog name from widgets. parmaterized
# WE have used datatime inorder to know the file was created on which date
import requests
import json
from datetime import datetime

url = f"{base_url}/summary/all_day.geojson"
response = requests.get(url)  # Fetch JSON data
if response.status_code != 200:    #Check if the request was successful (Status Code 200) 
    raise Exception(f"Request failed with status code {response.status_code}")
data = response.json() # COnverts into Python dictionary
current_date = datetime.now().strftime("%Y-%m-%d") # Fetch current date in YYYY-MM-DD format
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),  #Converts Python dictionary into JSON
    overwrite=True,
)